# 28.1 — Triplet Encoder + ANCE Hard Negative Mining (WJ 512)

Same encoder as nb28 but replaces **in-batch hard negatives** with **ANCE-style global hard negative mining**:
every `mine_every` epochs, encode the full corpus, build a temporary ANN index, and pick the
hardest non-GT corpus neighbor per query as an explicit negative. No B×B cross-similarity matrix —
each step is a clean (query, positive, hard_negative) triplet.

In [ ]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup, eval_recall, l1_simplex, load_dataset, load_dataset_normalized,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

dataset_name  = "full"
out_dim       = 512
device        = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
THREADS       = 40
seed          = 42
batch_size    = 2048
epochs        = 75
lr            = 1e-3
weight_decay  = 1e-4
max_pos       = 30
temperature   = 0.07
eval_every    = 15         # mid-training R@50 check
candidate_ks  = [500, 1000] if dataset_name == "10k" else [1000, 2000]

es_patience  = 25
es_min_delta = 1e-4

METHOD_NAME   = "triplet_ance_wj_512"
NOTEBOOK_NAME = "28_1_triplet_ance_wj_512.ipynb"
OUT_PATH      = "/tmp/results_sota_triplet_ance_wj_512.pkl"
CKPT_PATH     = "/tmp/best_sota_triplet_ance_wj_512_full.pt"

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"dataset={dataset_name} | batch={batch_size} | epochs={epochs} | temp={temperature} | eval_every={eval_every}")

In [2]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums, qt_norm = load_dataset_normalized(dataset_name)

dataset=full | qt=(233773, 18220) | corpus=(187019, 18220) | queries=(46754, 18220)
qt_norm loaded from cache (233773, 18220) in 11.9s


In [ ]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def wj_sim_matrix(a, b, chunk=256):
    """Chunked (A,B) WJ matrix — avoids materializing full (A,B,D) tensor at once."""
    rows = []
    for i in range(0, len(a), chunk):
        ai   = a[i:i+chunk]
        mins = torch.minimum(ai.unsqueeze(1), b.unsqueeze(0)).sum(-1)
        maxs = torch.maximum(ai.unsqueeze(1), b.unsqueeze(0)).sum(-1).clamp(1e-10)
        rows.append(mins / maxs)
    return torch.cat(rows, dim=0)

def inbatch_infonce_loss(zq, zp, temperature=0.07):
    """InfoNCE with all B in-batch items as negatives.
    Label i = zp[i] is the positive for zq[i]. 2047 negatives per query."""
    sim    = wj_sim_matrix(zq, zp) / temperature   # (B, B)
    labels = torch.arange(len(zq), device=zq.device)
    loss   = F.cross_entropy(sim, labels)
    with torch.no_grad():
        acc = (sim.argmax(1) == labels).float().mean().item()
    return loss, acc

class PairDataset(Dataset):
    def __init__(self, gt_lookup, query_start, max_pos=30):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            if qid < query_start: continue
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"  pairs={len(self.pairs):,} | steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        return torch.tensor(qid, dtype=torch.long), torch.tensor(pid, dtype=torch.long)

class TripletEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)

def embed_all(model, qt, batch_size=4096):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(model(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]; query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        cand, ci = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
        t0 = time.time()
        rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps_total = len(query_qt) / max(time.time()-t0 + len(query_qt)/max(ci['qps'],1e-9), 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"{key} R@{k} = {v:.4f}")
        print(f"{key} QPS={rr_metrics['qps']:.1f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})
    release_rerank_corpus()

In [4]:
device      = torch.device("cuda:0")
vecs_device = torch.device("cuda:7")
print("Pre-loading vectors to cuda:7...")
vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(vecs_device)
print(f"Loaded: {vecs_gpu.nbytes/1024**3:.2f} GB on {vecs_device}")

model = TripletEncoder(qt_norm.shape[1], out_dim)
model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
model = model.to(device)
print(f"DataParallel on {torch.cuda.device_count()} GPUs")

opt       = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
sch       = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
best_loss = float('inf')
no_imp    = 0
t0_train  = time.time()

# ── Phase 1: Warmup — in-batch InfoNCE, no ANCE mining ──────────────────────
# Gives model a head start before first mining. Avoids cold-start cascade where
# random model → random negatives → model trained on garbage → never recovers.
print("\n=== Phase 1: Warmup (in-batch InfoNCE, no mining) ===")
warmup_ds     = PairDataset(gt, query_start, max_pos=max_pos)
warmup_loader = DataLoader(warmup_ds, batch_size=batch_size, shuffle=True,
                           num_workers=4, pin_memory=True, drop_last=True)

for epoch in range(1, warmup_epochs + 1):
    model.train()
    tot_loss = tot_acc = steps = 0
    for q_ids, p_ids in tqdm(warmup_loader, desc=f"warmup ep{epoch:02d}", leave=False):
        all_ids  = torch.cat([q_ids, p_ids]).to(vecs_device)
        all_vecs = vecs_gpu[all_ids].to(device)
        B        = q_ids.shape[0]
        z        = model(all_vecs)
        zq, zp   = z[:B], z[B:]
        loss, acc = inbatch_infonce_loss(zq, zp, temperature=temperature)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot_loss += float(loss.detach()); tot_acc += acc; steps += 1
    sch.step()
    avg     = tot_loss / max(steps, 1)
    elapsed = (time.time() - t0_train) / 60
    print(f"warmup ep{epoch:02d}/{warmup_epochs} | loss={avg:.4f} | acc={tot_acc/max(steps,1):.3f} | {elapsed:.1f}min", flush=True)
    if avg < best_loss - es_min_delta:
        best_loss = avg
        torch.save(model.module.state_dict(), CKPT_PATH)

# ── Phase 2: ANCE — hybrid InfoNCE (in-batch + mined hard negatives) ─────────
print("\n=== Phase 2: ANCE + Hybrid InfoNCE ===")
print(f"[init] Mining after warmup (k={mine_k}, n={n_hard_negs}):", flush=True)
hard_negs = mine_hard_negatives(model, qt_norm, gt, query_start)

def make_ance_loader(hard_negs):
    ds = ANCEDataset(gt, query_start, hard_negs, max_pos=max_pos, n_negs=n_hard_negs)
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True)

ance_loader = make_ance_loader(hard_negs)
no_imp      = 0

epoch_bar = tqdm(range(warmup_epochs + 1, epochs + 1), desc="ANCE epochs", unit="ep")
for epoch in epoch_bar:
    # Re-mine every mine_every epochs starting from epoch warmup+1+mine_every
    ance_ep = epoch - warmup_epochs   # 1-indexed within ANCE phase
    if ance_ep > 1 and (ance_ep - 1) % mine_every == 0:
        print(f"\n[ep{epoch:02d}] Re-mining (k={mine_k}, n={n_hard_negs}):", flush=True)
        hard_negs   = mine_hard_negatives(model, qt_norm, gt, query_start)
        ance_loader = make_ance_loader(hard_negs)

    # Mid-training diagnostic: quick R@50 to catch collapse early
    if epoch % eval_every == 0:
        print(f"\n[ep{epoch:02d}] Quick eval...", flush=True)
        model.eval()
        embs_all = embed_all(model, qt_norm)
        model.train()
        c_eval, q_eval = embs_all[:query_start], embs_all[query_start:]
        nbrs_eval, _   = nmslib_neighbors(c_eval, q_eval, space="WeightedJaccard", k=50, threads=THREADS)
        m = eval_recall(gt, nbrs_eval, query_start, 50)
        print(f"  [ep{epoch:02d}] R@10={m[10]:.4f}  R@50={m[50]:.4f}", flush=True)
        del embs_all, c_eval, q_eval, nbrs_eval

    model.train()
    tot_loss = tot_acc = steps = 0
    step_bar = tqdm(ance_loader, desc=f"ep{epoch:02d}", leave=False, unit="step")
    for q_ids, p_ids, n_ids_batch in step_bar:
        B, N = q_ids.shape[0], n_ids_batch.shape[1]
        # Load q, p, and all N hard negatives in one indexed fetch from cuda:7
        all_ids  = torch.cat([q_ids, p_ids] + [n_ids_batch[:, i] for i in range(N)]).to(vecs_device)
        all_vecs = vecs_gpu[all_ids].to(device)            # (B*(2+N), D_in)
        z        = model(all_vecs)                          # (B*(2+N), out_dim)
        zq       = z[:B]
        zp       = z[B:2*B]
        z_hard   = torch.stack([z[(2+i)*B:(3+i)*B] for i in range(N)], dim=1)  # (B, N, out_dim)
        loss, acc = hybrid_infonce_loss(zq, zp, z_hard, temperature=temperature)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot_loss += float(loss.detach()); tot_acc += acc; steps += 1
        step_bar.set_postfix(loss=f"{float(loss.detach()):.4f}", acc=f"{acc:.3f}")
    sch.step()

    avg     = tot_loss / max(steps, 1)
    avg_acc = tot_acc  / max(steps, 1)
    elapsed = (time.time() - t0_train) / 60
    eta     = elapsed / max(epoch, 1) * (epochs - epoch)
    epoch_bar.set_postfix(loss=f"{avg:.4f}", best=f"{best_loss:.4f}", acc=f"{avg_acc:.3f}", eta=f"{eta:.0f}m")
    if epoch % 5 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} | loss={avg:.4f} | acc={avg_acc:.3f} | "
              f"best={best_loss:.4f} | {elapsed:.1f}min | eta={eta:.1f}min", flush=True)

    if avg < best_loss - es_min_delta:
        best_loss = avg; no_imp = 0
        torch.save(model.module.state_dict(), CKPT_PATH)
    else:
        no_imp += 1
        if no_imp >= es_patience:
            print(f"\nEarly stop ep{epoch}: no improvement for {es_patience} epochs", flush=True)
            break

print(f"\nTraining done. best_loss={best_loss:.4f} | ckpt={CKPT_PATH}")

Pre-loading vectors to cuda:7...
Loaded: 15.87 GB on cuda:7
DataParallel on 8 GPUs

=== Phase 1: Warmup (in-batch InfoNCE, no mining) ===
  warmup pairs=1,285,479 | steps/epoch=627


warmup ep01:   0%|          | 0/627 [00:00<?, ?it/s]

/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return F.linear(input, self.weight, self.bias)


warmup ep01/10 | loss=0.9675 | acc=0.835 | 3.1min


warmup ep02:   0%|          | 0/627 [00:00<?, ?it/s]

...

ep29:   0%|          | 0/625 [00:00<?, ?step/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f541d9583a0>
Traceback (most recent call last):
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
Exception ignored in: AssertionError: can only test a child process
<function _MultiProcessingDataLoaderIter.__del__ at 0x7f541d9583a0>
Traceback (most recent call last):
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fil

ep30:   0%|          | 0/625 [00:00<?, ?step/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f541d9583a0>
Traceback (most recent call last):
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f541d9583a0>
Traceback (most recent call last):
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fil

epoch 30/100 | loss=1.0464 | acc=0.722 | best=0.6700 | 120.3min | eta=280.8min

Early stop ep30: no improvement for 20 epochs

Training done. best_loss=0.6700 | ckpt=/tmp/best_sota_triplet_ance_wj_512_full.pt


In [7]:
(model.module if hasattr(model, "module") else model).load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

R@10   = 0.5334
R@50   = 0.6307
R@100  = 0.6541
R@500  = 0.7096
QPS=3028.6
saved triplet_ance_wj_512 -> /tmp/results_sota_triplet_ance_wj_512.pkl
Corpus pre-loaded to GPU: 12.69 GB



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*******************************************************

triplet_ance_wj_512_rerank_1000 R@10 = 0.9921
triplet_ance_wj_512_rerank_1000 R@50 = 0.9915
triplet_ance_wj_512_rerank_1000 R@100 = 0.9832
triplet_ance_wj_512_rerank_1000 R@500 = 0.8389
triplet_ance_wj_512_rerank_1000 QPS=1222.5
saved triplet_ance_wj_512_rerank_1000 -> /tmp/results_sota_triplet_ance_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

triplet_ance_wj_512_rerank_2000 R@10 = 0.9922
triplet_ance_wj_512_rerank_2000 R@50 = 0.9920
triplet_ance_wj_512_rerank_2000 R@100 = 0.9849
triplet_ance_wj_512_rerank_2000 R@500 = 0.8512
triplet_ance_wj_512_rerank_2000 QPS=872.4
saved triplet_ance_wj_512_rerank_2000 -> /tmp/results_sota_triplet_ance_wj_512.pkl


In [8]:

# ── GPU Exact L1 Search ───────────────────────────────────────────────────────
# Diagnoses whether HNSW efSearch is the bottleneck or if embeddings themselves
# are the problem. WJ on L1-simplex is monotone with L1 distance, so exact L1
# nearest-neighbor search gives the true recall ceiling of this embedding space.

expected_rows = len(qt_norm)  # 233773 for full, 10000 for 10k
if 'embs' not in vars() or embs.shape[0] != expected_rows:
    print(f"Re-encoding (embs missing or wrong shape — expected {expected_rows} rows)...")
    enc = TripletEncoder(qt_norm.shape[1], out_dim)
    enc.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
    enc = nn.DataParallel(enc, device_ids=list(range(torch.cuda.device_count()))).to(device)
    embs = embed_all(enc, qt_norm)
    print(f"embs: {embs.shape}")
else:
    print(f"Using existing embs {embs.shape}")

corpus_embs_ex = embs[:query_start]
query_embs_ex  = embs[query_start:]
N, D = corpus_embs_ex.shape
Q    = len(query_embs_ex)
k    = 50

print(f"Exact search: {Q} queries × {N} corpus | D={D} | k={k}")
search_dev = torch.device("cuda:0")
corpus_gpu_ex = torch.from_numpy(corpus_embs_ex).to(search_dev)
print(f"Corpus on GPU: {corpus_gpu_ex.nbytes/1024**3:.2f} GB")

chunk_size = 64   # 64×187K×512×4 ≈ 23 GB intermediate — fine on A100
all_idx = []
t0 = time.time()
with torch.no_grad():
    for s in range(0, Q, chunk_size):
        q = torch.from_numpy(query_embs_ex[s:s+chunk_size]).to(search_dev)
        l1 = (q.unsqueeze(1) - corpus_gpu_ex.unsqueeze(0)).abs_().sum(dim=2)
        all_idx.append(l1.topk(k, dim=1, largest=False).indices.cpu().numpy())
elapsed = time.time() - t0
qps_exact = Q / elapsed
nbrs_exact = np.vstack(all_idx)

metrics_ex = eval_recall(gt, nbrs_exact, query_start, k)
print(f"\n--- GPU Exact L1 ---")
for kk in [10, 50]:
    print(f"R@{kk:<4} = {metrics_ex[kk]:.4f}")
print(f"QPS  = {qps_exact:.0f}  ({elapsed:.1f}s total)")
print(f"\nHNSW R@10=0.2919  →  Exact R@10={metrics_ex[10]:.4f}")
print("If Exact ≈ HNSW: problem is embedding quality, not search approximation.")
print("If Exact >> HNSW: efSearch too small, increase it.")
del corpus_gpu_ex


Using existing embs (233773, 512)
Exact search: 46754 queries × 187019 corpus | D=512 | k=50
Corpus on GPU: 0.36 GB

--- GPU Exact L1 ---
R@10   = 0.5334
R@50   = 0.6307
QPS  = 848  (55.1s total)

HNSW R@10=0.2919  →  Exact R@10=0.5334
If Exact ≈ HNSW: problem is embedding quality, not search approximation.
If Exact >> HNSW: efSearch too small, increase it.
